In [1]:
from xgboost import XGBClassifier

from intrusion_classifier import (
    aggregate_feature_importance,
    build_pipeline,
    evaluate_predictions,
    load_data,
    save_pipeline,
    select_features_by_importance,
    split_data,
)

features, target = load_data()
splits = split_data(features, target)

xgboost = build_pipeline(
    XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric="logloss",
        n_jobs=1,
    ),
    splits.X_train,
)
xgboost.fit(splits.X_train, splits.y_train)

predictions = xgboost.predict(splits.X_test)
evaluate_predictions(splits.y_test, predictions, model_name="XGBoost")
save_pipeline(xgboost, "artifacts/xgboost.joblib")

feature_importance = aggregate_feature_importance(xgboost)
selected_features = select_features_by_importance(feature_importance, threshold=0.95)
print("\nTop 10 important source features:")
print(feature_importance.head(10).to_string())
print("\nFeatures accounting for at least 95% of importance:")
print(selected_features.index.tolist())
print("Cumulative importance:", selected_features.sum())


XGBoost test accuracy: 0.997023
              precision    recall  f1-score   support

      normal     0.9959    0.9985    0.9972      2690
     anomaly     0.9983    0.9953    0.9968      2349

    accuracy                         0.9970      5039
   macro avg     0.9971    0.9969    0.9970      5039
weighted avg     0.9970    0.9970    0.9970      5039


Top 10 important source features:
service                        0.534578
src_bytes                      0.158019
wrong_fragment                 0.062178
logged_in                      0.050336
flag                           0.033035
hot                            0.027823
dst_host_same_srv_rate         0.019458
dst_bytes                      0.015769
count                          0.012729
dst_host_same_src_port_rate    0.011895

Features accounting for at least 95% of importance:
['service', 'src_bytes', 'wrong_fragment', 'logged_in', 'flag', 'hot', 'dst_host_same_srv_rate', 'dst_bytes', 'count', 'dst_host_same_src_port_rate', 'sr

/Users/cooperstepanian/dev/Intrusion-classifier/intrusion_classifier.py:141: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
